In [ ]:
import json
import pandas as pd
import hashlib

# Ensure the ADOMD.NET path is in the system path prior to importing Pyadomd
from sys import path
adomd_path = r'C:\Program Files\Microsoft.NET\ADOMD.NET\160'
if adomd_path not in path:
    path.append(adomd_path)

# for x in path:
#     print(x, sep='\n')

from pyadomd import Pyadomd

json_path = "Sales & Returns Sample v201912.json"

with open(json_path, "r", encoding="utf-8-sig") as f:
    data = json.load(f)

# events = data["events"]
events = data.get("events", [])

# Build a lookup dictionary for fast id -> node access
id_map = {event['id']: event for event in events if 'id' in event}

def find_visual_id(id_map, start_id):
    current_id = start_id
    while current_id:
        node = id_map.get(current_id)
        if not node:
            break
        metrics = node.get('metrics', {})
        visual_id = metrics.get('visualId')
        if visual_id:
            return visual_id
        current_id = node.get('parentId')
    return None

# Flatten metrics into top-level columns
def flatten_event(event):
    flat = event.copy()
    metrics = flat.pop("metrics", {})
    flat.update(metrics)
    return flat

flat_events = [flatten_event(e) for e in data.get("events", [])]

df = pd.DataFrame(flat_events)

# Apply to each row in the DataFrame to get a new column with the visualId
df['visualId'] = df['id'].apply(lambda x: find_visual_id(id_map, x))

pd.set_option('display.max_columns', None)  # Show all columns
pd.set_option('display.max_rows', 20)    
#print(df.head())

filtered_df = df[df['name'] == 'Execute DAX Query'].copy()
filtered_df.drop_duplicates(inplace=True)

# filtered_df = df[
#     (df['name'] == 'Execute DAX Query') &
#     (df['id'] == 'baa3553b-c0fa-4f0e-87d6-98ea260fecba')
# ].copy()
filtered_df


# Connection string to your Power BI semantic model (update as needed)
connection_string = "Provider=MSOLAP;Data Source=localhost:61954;Initial Catalog=31499e00-1fda-40a9-b59e-a2227c83ab3d"


def row_hash(row):
    # Concatenate all values as string and hash
    row_str = '|'.join(str(val) for val in row.values)
    return hashlib.sha256(row_str.encode('utf-8')).hexdigest()


# Loop through the DataFrame and execute each DAX query
with Pyadomd(connection_string) as conn:
    #Loop over each query from the dataframe
    for idx, row in filtered_df.iterrows():
        query_id = row['id']
        query_text = row['QueryText']
        print(f"QueryText:\n{query_text[:50]}\n")
        #print(f"Executing Query ID: {query_id}")

        try:
            with conn.cursor().execute(query_text) as cur:
                result_set_index = 1
                query_hashes = []  # List to collect all query hashes
                has_next = True
                while has_next:
                    columns = [col[0] for col in cur.description]
                    result_rows = cur.fetchall()
                    print(f"Result set {result_set_index} has {len(result_rows)} rows")

                    if result_rows:
                        # Create a DataFrame for this query's results
                        result_df = pd.DataFrame(result_rows, columns=columns)

                        # Add a hash column for each row
                        if not result_df.empty:
                            result_df['row_hash'] = result_df.apply(row_hash, axis=1)

                            # Sort by row_hash
                            result_df = result_df.sort_values('row_hash').reset_index(drop=True)
                            # Concatenate all row_hash values and hash them
                            row_hashes = '|'.join(result_df['row_hash'].tolist())
                            combined_row_hash = hashlib.sha256(row_hashes.encode('utf-8')).hexdigest()

                            query_hashes.append(combined_row_hash)

                            # Store in filtered_df (for the first result set only, or adjust as needed)
                            # if result_set_index == 0:
                            #     filtered_df.loc[idx, 'combined_row_hash'] = combined_row_hash
                            print(f"query: {query_id} result set {result_set_index} rows: {len(result_df)} combined hash: {combined_row_hash}")
                        else:
                            print(f"query: {query_id} result set {result_set_index} is empty")

                    result_set_index += 1
                    has_next = cur.nextresult()

                if query_hashes:
                    combined = '|'.join(query_hashes)
                    combined_query_hash = hashlib.sha256(combined.encode('utf-8')).hexdigest()
                    print(f"Final combined hash: {combined_query_hash}")

                    # Store in filtered_df (for the first result set only, or adjust as needed)
                    filtered_df.loc[idx, 'combined_query_hash'] = combined_query_hash

        except Exception as e:
            print(f"Error executing query {query_id}: {e}\n")

filtered_df
# filtered_df.to_csv("filtered_df.csv", index=False)

QueryText:
DEFINE VAR __DS0FilterTable = 
	TREATAS({"June"},

Result set 1 has 1 rows
query: 82bd947a-c0cb-48f5-87aa-2db65515b25e result set 1 rows: 1 combined hash: dfaa90b9f1d0497f2836e7a73b6e332125ba378ff023382491970708da76b0d5
Final combined hash: bf3b72873b105c202bd18b2025d0b6becbcd63c5ab45b04704efd50dd91ed9f4
QueryText:
DEFINE
	VAR __DS0FilterTable = 
		TREATAS({"June

Result set 1 has 1 rows
query: baa3553b-c0fa-4f0e-87d6-98ea260fecba result set 1 rows: 1 combined hash: dfaa90b9f1d0497f2836e7a73b6e332125ba378ff023382491970708da76b0d5
Result set 2 has 33 rows
query: baa3553b-c0fa-4f0e-87d6-98ea260fecba result set 2 rows: 33 combined hash: f05c4829ddfeca46de5f15ca46f2c09f8efad35d27b681ce5934562a0522ea7a
Final combined hash: f337c81e916a742acf92fa3a99a94d5b2fa616fe79311f81e3e411c2ebce641b
QueryText:
DEFINE
	VAR __DS0FilterTable = 
		TREATAS({"Sold

Result set 1 has 1 rows
query: 09fa85a8-b263-484c-a590-1fb11a89288a result set 1 rows: 1 combined hash: a587cdfbde2b7d9e4235afdeadd82d6

,name,component,start,id,sourceLabel,end,status,visualTitle,visualId,visualType,initialLoad,parentId,QueryText,RowCount,combined_query_hash
37,Execute DAX Query,DSE,2025-05-28T16:12:14.236Z,82bd947a-c0cb-48f5-87aa-2db65515b25e,NaN,2025-05-28T16:12:14.249Z,NaN,NaN,a03ed496f51eef7b5c23,NaN,NaN,14e4cc77-0fae-44fa-90db-9708a86ef10f,"DEFINE VAR __DS0FilterTable = \r\n\tTREATAS({""...",1.0,bf3b72873b105c202bd18b2025d0b6becbcd63c5ab45b0...
69,Execute DAX Query,DSE,2025-05-28T16:12:14.236Z,baa3553b-c0fa-4f0e-87d6-98ea260fecba,NaN,2025-05-28T16:12:14.249Z,NaN,NaN,94e00a62024369b7a40e,NaN,NaN,44f14ad4-8cdf-4d6b-9574-d8c4cbdc5e60,DEFINE\r\n\tVAR __DS0FilterTable = \r\n\t\tTRE...,34.0,f337c81e916a742acf92fa3a99a94d5b2fa616fe79311f...
98,Execute DAX Query,DSE,2025-05-28T16:12:17.026Z,09fa85a8-b263-484c-a590-1fb11a89288a,NaN,2025-05-28T16:12:17.034Z,NaN,NaN,327c785bec894bb22e8a,NaN,NaN,e2c0fd11-2c16-42a2-ba70-68c8bfe0eb41,DEFINE\r\n\tVAR __DS0FilterTable = \r\n\t\tTRE...,1.0,46c55ee3ab6c6163ef2b6ebaf8971b305bb55271c9244d...
111,Execute DAX Query,DSE,2025-05-28T16:12:17.051Z,206771da-ca77-40ad-a75b-280d0d7a62eb,NaN,2025-05-28T16:12:17.057Z,NaN,NaN,fd225c7a3a3a0ea6006c,NaN,NaN,ba8a86ad-88b7-4d0e-a6cc-e76c8ae7479f,DEFINE\r\n\tVAR __DS0FilterTable = \r\n\t\tTRE...,1.0,1c08ffdf06410e3a608a3c0ebb2d4a2c1862ff83a5f676...
128,Execute DAX Query,DSE,2025-05-28T16:12:17.051Z,9f6684fc-b91b-4b9a-9650-4edb74369753,NaN,2025-05-28T16:12:17.061Z,NaN,NaN,dec3aad399d5d19a1d11,NaN,NaN,be22ce78-faee-4c00-bf3f-fa40afb7547c,DEFINE\r\n\tVAR __DS0FilterTable = \r\n\t\tTRE...,1.0,916d488c3afa83e802a039f1241264094a79ac0ed59bb0...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
664,Execute DAX Query,DSE,2025-05-28T16:12:28.714Z,e033aaf7-a2aa-413f-8a2f-969a4deb4615,NaN,2025-05-28T16:12:28.730Z,NaN,NaN,e5b7b6b90231b41809a1,NaN,NaN,d96a4290-a4a6-4b4e-9ba8-7da7a725cb34,DEFINE\r\n\tVAR __DS0FilterTable = \r\n\t\tFIL...,1.0,a3e30f2f2b65e66d1016efa882c59db7b1b8cf50503758...
678,Execute DAX Query,DSE,2025-05-28T16:12:28.715Z,c8f382ac-e545-4879-bd35-67f20096b8d7,NaN,2025-05-28T16:12:28.730Z,NaN,NaN,948988db07a4db09d58d,NaN,NaN,8e712962-ac31-485e-8c04-c0d576955b18,DEFINE\r\n\tVAR __DS0FilterTable = \r\n\t\tFIL...,26.0,0fea39528cc2f3bfc39d83d0a11d892a376f194ade4cf4...
694,Execute DAX Query,DSE,2025-05-28T16:12:28.714Z,de5a7a49-e8bd-4659-8f45-a32f6eb2a076,NaN,2025-05-28T16:12:28.730Z,NaN,NaN,031aa35ec577ca25a558,NaN,NaN,3506cac7-3be4-4c57-96cf-77cebe329334,DEFINE\r\n\tVAR __DS0FilterTable = \r\n\t\tFIL...,1.0,bf3b72873b105c202bd18b2025d0b6becbcd63c5ab45b0...
760,Execute DAX Query,DSE,2025-05-28T16:12:28.734Z,20e49a5f-3c16-4761-a702-2e3b936b60b0,NaN,2025-05-28T16:12:28.740Z,NaN,NaN,0985c85625605520715b,NaN,NaN,ddaf755a-216b-48e8-b284-cd51c8141f83,DEFINE\r\n\tVAR __DS0FilterTable = \r\n\t\tFIL...,1.0,de5ff8bbe3acfc48f40348489c017bbba7846562babdf8...
